In [ ]:
!pip install anthropic scikit-learn pandas numpy -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.2/458.2 kB 5.4 MB/s eta 0:00:00


In [ ]:
import anthropic
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

client = anthropic.Anthropic(api_key="")

def run_agent(system_prompt, user_message, max_tokens=600):
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text

print("Setup complete.")

Setup complete.


In [ ]:
import pandas as pd

# Load the sheets we need
xl = pd.ExcelFile("/content/neobank_fintech_dataset.xlsx")

df_pl      = pd.read_excel(xl, sheet_name="P&L",                header=2)
df_kpi     = pd.read_excel(xl, sheet_name="KPIs",               header=2)
df_bva     = pd.read_excel(xl, sheet_name="Budget vs Actuals",  header=2)
df_loans   = pd.read_excel(xl, sheet_name="Loan Portfolio",     header=2)

# Quick sanity check
print("P&L shape:",     df_pl.shape)
print("KPI shape:",     df_kpi.shape)
print("BvA shape:",     df_bva.shape)
print("Loans shape:",   df_loans.shape)

print("\n--- P&L columns ---")
print(df_pl.columns.tolist())

P&L shape: (32, 14)
KPI shape: (31, 14)
BvA shape: (16, 7)
Loans shape: (8, 9)

--- P&L columns ---
['Line Item', 'Jan-24', 'Feb-24', 'Mar-24', 'Apr-24', 'May-24', 'Jun-24', 'Jul-24', 'Aug-24', 'Sep-24', 'Oct-24', 'Nov-24', 'Dec-24', 'FY 2024']


In [ ]:
# Set Line Item as index for easy row lookup
df_pl.set_index("Line Item", inplace=True)
df_kpi.set_index("KPI", inplace=True)
df_bva.set_index("Metric", inplace=True)

month_cols = ['Jan-24','Feb-24','Mar-24','Apr-24','May-24','Jun-24',
              'Jul-24','Aug-24','Sep-24','Oct-24','Nov-24','Dec-24']

# Pull key P&L rows
revenue  = df_pl.loc["Total Revenue",      month_cols]
ebitda   = df_pl.loc["EBITDA",             month_cols]
net_inc  = df_pl.loc["Net Income / (Loss)",month_cols]

# Pull key KPIs
users    = df_kpi.loc["  Active Users (000s)", month_cols]
cac      = df_kpi.loc["  CAC ($)",             month_cols]
ltv_cac  = df_kpi.loc["  LTV:CAC Ratio",       month_cols]
npl      = df_kpi.loc["  90+ DPD (NPL) Rate",  month_cols]

# Format for agent consumption
pl_summary = pd.DataFrame({
    "Revenue":  revenue.values,
    "EBITDA":   ebitda.values,
    "Net Inc":  net_inc.values,
    "Users(k)": users.values,
    "CAC($)":   cac.values,
    "LTV:CAC":  ltv_cac.values,
    "NPL%":     npl.values
}, index=month_cols).round(2)

print(pl_summary.to_string())

        Revenue  EBITDA  Net Inc Users(k) CAC($) LTV:CAC   NPL%
Jan-24   4200.0   739.2    571.2    310.0   48.0     8.8  0.012
Feb-24   4350.0   765.6    591.6    328.0   46.0     9.5  0.011
Mar-24   4180.0   735.7    568.5    342.0   50.0     8.6  0.013
Apr-24   4620.0   813.1    628.3    361.0   44.0    10.1   0.01
May-24   4800.0   844.8    652.8    385.0   42.0    11.0   0.01
Jun-24   5100.0   897.6    693.6    412.0   40.0    12.0  0.009
Jul-24   5250.0   924.0    714.0    438.0   39.0    12.6  0.008
Aug-24   4100.0   721.6    557.6    421.0   65.0     6.3  0.016
Sep-24   5400.0   950.4    734.4      455     38    13.3  0.008
Oct-24   5650.0   994.4    768.4      482     36    14.4  0.007
Nov-24   5900.0  1038.4    802.4      510     35    15.4  0.007
Dec-24   6100.0  1073.6    829.6      545     33    16.8  0.006


In [ ]:
import anthropic
client = anthropic.Anthropic(api_key="")

def run_agent(system_prompt, user_message, max_tokens=600):
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text

data_str = pl_summary.to_string()

# Agent 1 — Fintech Analyst
analyst_out = run_agent(
    system_prompt="""You are a senior fintech analyst specialising in neobanks.
    Analyse the monthly data and extract key trends across revenue, profitability,
    unit economics, and credit quality. Be specific about numbers. Flag anomalies.""",
    user_message=f"Analyse this NeoBank monthly dataset:\n\n{data_str}"
)
print("=== AGENT 1: FINTECH ANALYST ===\n")
print(analyst_out)

=== AGENT 1: FINTECH ANALYST ===

# NeoBank Monthly Performance Analysis: FY2024

## Executive Summary

Strong growth trajectory with one significant anomaly in August. Full-year revenue of ~$59.7M with consistent margin expansion and improving unit economics. Credit quality remains excellent throughout.

---

## 1. Revenue & Profitability Trends

### Revenue Growth
| Metric | Jan-24 | Dec-24 | Change |
|--------|--------|--------|--------|
| Revenue | $4,200K | $6,100K | **+45.2%** |
| EBITDA | $739K | $1,074K | **+45.3%** |
| Net Income | $571K | $830K | **+45.3%** |

**Key Observations:**
- Consistent month-over-month growth averaging **~3.4%** (excluding August)
- EBITDA margin stable at **~17.6%** throughout the year
- Net margin steady at **~13.6%** — indicates disciplined cost structure

---

## 2. User Growth & Unit Economics

### User Acquisition
- **Jan-24:** 310K users → **Dec-24:** 545K users
- **Net adds:** 235K users (+75.8% YoY)
- **Average monthly adds:** ~19.6K users



In [ ]:
# Agent 2 — Credit & Risk Officer
risk_out = run_agent(
    system_prompt="""You are a Chief Risk Officer at a neobank.
Review the financial and KPI analysis and identify credit risks,
unit economics deterioration, and operational red flags.
Prioritise by severity. Ground every point in the numbers.""",
    user_message=f"Identify key risks from this analysis:\n\n{analyst_out}"
)

# Agent 3 — CFO Board Briefing
cfo_out = run_agent(
    system_prompt="""You are the CFO of a neobank presenting to the board.
Synthesise the analysis and risk review into a board-ready briefing.
Structure: 1) Headline (2 sentences), 2) Top 3 achievements,
3) Top 3 risks, 4) Recommended actions with owners and deadlines.
Be decisive, boards need clarity not caveats.""",
    user_message=f"ANALYSIS:\n{analyst_out}\n\nRISK REVIEW:\n{risk_out}",
    max_tokens=700
)

sep = "=" * 60
print(sep)
print("NEOBANK INC. — FY2024 BOARD BRIEFING")
print(sep)
print("\nFINTECH ANALYST FINDINGS\n")
print(analyst_out)
print("\nRISK REVIEW\n")
print(risk_out)
print("\nCFO BOARD BRIEFING\n")
print("-" * 60)
print(cfo_out)
print(sep)

NEOBANK INC. — FY2024 BOARD BRIEFING

FINTECH ANALYST FINDINGS

# NeoBank Monthly Performance Analysis: FY2024

## Executive Summary

Strong growth trajectory with one significant anomaly in August. Full-year revenue of ~$59.7M with consistent margin expansion and improving unit economics. Credit quality remains excellent throughout.

---

## 1. Revenue & Profitability Trends

### Revenue Growth
| Metric | Jan-24 | Dec-24 | Change |
|--------|--------|--------|--------|
| Revenue | $4,200K | $6,100K | **+45.2%** |
| EBITDA | $739K | $1,074K | **+45.3%** |
| Net Income | $571K | $830K | **+45.3%** |

**Key Observations:**
- Consistent month-over-month growth averaging **~3.4%** (excluding August)
- EBITDA margin stable at **~17.6%** throughout the year
- Net margin steady at **~13.6%** — indicates disciplined cost structure

---

## 2. User Growth & Unit Economics

### User Acquisition
- **Jan-24:** 310K users → **Dec-24:** 545K users
- **Net adds:** 235K users (+75.8% YoY)
- **Average 